# Chapter 02 — XML, DTD, WIPO ST.36 and ST.96

*Where we are:* **parsing** — turning the structured source (XML) into our document model.

```
[ XML / ST.36 / ST.96 ]→ document model → normalization → chunking → retrieval
```

Patents are exchanged as XML. To ingest them faithfully you must understand XML structure,
**validation** (DTD and XSD), and the two WIPO standards you will actually meet: legacy
**ST.36** (DTD-based) and current **ST.96** (XSD-based). We work on a **genuine WIPO ST.96
example instance** — real US Patent 8,936,998 B2.

In [1]:
# === Chapter 02 · standard bootstrap (identical pattern in every notebook) ===
# Runs standalone on a fresh Google Colab VM *or* a local checkout.
import os, sys, subprocess

REPO_URL = "https://github.com/rsalehin/patent-rag-masterclass"
NEED_OCR = False
IN_COLAB = "google.colab" in sys.modules


def _clone_repo(url, target):
    """Clone the repo on Colab. For a PRIVATE repo, authenticate with a GitHub token read from
    Colab Secrets (key 'GITHUB_TOKEN') or the GITHUB_TOKEN env var. The token is never printed."""
    token = None
    try:
        from google.colab import userdata  # type: ignore
        token = userdata.get("GITHUB_TOKEN")
    except Exception:
        token = os.environ.get("GITHUB_TOKEN")
    auth_url = url
    if token and url.startswith("https://github.com/"):
        auth_url = url.replace("https://github.com/", f"https://{token}@github.com/")
    r = subprocess.run(["git", "clone", "--depth", "1", auth_url, target],
                       stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)  # avoid leaking the token
    if r.returncode != 0:
        raise RuntimeError(
            "git clone failed. This is a PRIVATE repo, so Colab needs a GitHub token:\n"
            "  1) Create a token (scope: repo) at https://github.com/settings/tokens\n"
            "  2) In Colab, open the key icon (Secrets) in the left sidebar, add a secret named\n"
            "     GITHUB_TOKEN, paste the token, and enable 'Notebook access'.\n"
            "  3) Re-run this cell.\n"
            "  (Alternatively, make the GitHub repo public — then no token is needed.)")


if IN_COLAB:
    target = "/content/patent-rag-masterclass"
    if not os.path.isdir(target):
        if not REPO_URL:
            raise RuntimeError("Set REPO_URL to this repo's GitHub URL (see README.md).")
        _clone_repo(REPO_URL, target)
    os.chdir(target)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    if NEED_OCR:
        subprocess.run(["apt-get", "install", "-y", "-q", "tesseract-ocr"], check=False)

# Ensure the repo root (containing patentrag/) is importable.
for _cand in [os.getcwd()] + [os.path.dirname(os.getcwd())]:
    if os.path.isdir(os.path.join(_cand, "patentrag")):
        if _cand not in sys.path:
            sys.path.insert(0, _cand)
        break

from patentrag import bootstrap as bs
bs.setup_environment(REPO_URL, need_ocr=NEED_OCR)
bs.set_seeds()
_env = bs.environment_report()
print("Chapter 02 bootstrap OK")
print("  Python", _env["python"], "| Colab:", _env["in_colab"], "| CPU cores:", _env["cpu_count"])
print("  torch", _env["torch"], "| CUDA:", _env["cuda_available"], "| tesseract:", _env["tesseract"])

Chapter 02 bootstrap OK
  Python 3.12.10 | Colab: False | CPU cores: 24
  torch 2.12.0.dev20260304+cu130 | CUDA: True | tesseract: True


## 3. XML fundamentals

An XML document is a tree of **elements** (`<tag>…</tag>`) with **attributes**, optionally in
**namespaces** (a URI that disambiguates tag names). **XPath** is the query language over that
tree. **Entities** are named substitutions (`&amp;`), and a **schema** (DTD or XSD) constrains
which structures are valid. `lxml` gives us fast parsing + XPath + both validation styles.

In [2]:
from patentrag import parsing as P
tree = P.load_xml(bs.DATA / "xml" / "ST96_PatentPublication_Example.xml")
root = tree.getroot()
from lxml import etree
print("root element  :", etree.QName(root).localname)
print("namespaces    :")
for prefix, uri in root.nsmap.items():
    print(f"    {prefix or '(default)':8} -> {uri}")
print("total elements:", sum(1 for _ in root.iter()))

root element  : PatentPublication
namespaces    :
    pat      -> http://www.wipo.int/standards/XMLSchema/ST96/Patent
    xsi      -> http://www.w3.org/2001/XMLSchema-instance
    com      -> http://www.wipo.int/standards/XMLSchema/ST96/Common
total elements: 1509


### XPath extraction

ST.96 is namespaced (`pat:` = Patent components, `com:` = Common components). We extract
bibliographic fields and claims with namespace-aware XPath. This is the real patent's data.

In [3]:
import pandas as pd
bib = P.st96_bibliographic(tree)
pd.Series(bib).to_frame("value")

,value
publication_number,08936998
kind_code,B2
publication_date,2015-01-20
application_number,13797521
title,Manufacturing method for room-temperature subs...
ip_office,US
st96_version,V8_0


In [4]:
# XPath directly: pull all publication-reference dates via a namespaced query
dates = P.xpath(tree, "//com:PublicationDate/text()")
kinds = P.xpath(tree, "//com:PatentDocumentKindCode/text()")
print("PublicationDate nodes via XPath:", dates)
print("KindCode nodes via XPath       :", kinds[:5], "...")

claims = P.st96_claims(tree)
print(f"\nextracted {len(claims)} claims; claim 1 (head):")
print(" ", claims[0][:180])

PublicationDate nodes via XPath: ['2015-01-20', '2013-08-22']
KindCode nodes via XPath       : ['B2', 'A', 'B2', 'B2', 'A1'] ...

extracted 10 claims; claim 1 (head):
  1 Independent 1. A device manufacture method, comprising: sputtering a first surface of a first substrate mainly containing silicon dioxide; and preparing a bonded substrate by roo


## 4. DTDs (and the ST.36 lineage)

A **DTD** (Document Type Definition) is the older schema language: it declares elements, their
allowed children, and attributes, but has **no namespaces and weak typing** (everything is
essentially text). Older patent pipelines — and USPTO's ICE "red book" grant XML — are
DTD-based (ST.36 lineage). We demonstrate DTD validation with a compact, genuine-style ST.36
schema and a real patent's content.

In [5]:
print("Compact ST.36-style DTD (teaching subset of real ST.36 element names):\n")
print(P.ST36_DTD)

Compact ST.36-style DTD (teaching subset of real ST.36 element names):

<!ELEMENT patent-document (bibliographic-data, abstract?, description?, claims?)>
<!ATTLIST patent-document lang CDATA #REQUIRED country CDATA #REQUIRED
                          doc-number CDATA #REQUIRED kind CDATA #IMPLIED>
<!ELEMENT bibliographic-data (invention-title, publication-date?)>
<!ELEMENT invention-title (#PCDATA)>
<!ELEMENT publication-date (#PCDATA)>
<!ELEMENT abstract (p+)>
<!ELEMENT description (p+)>
<!ELEMENT claims (claim+)>
<!ELEMENT claim (claim-text)>
<!ATTLIST claim num CDATA #REQUIRED>
<!ELEMENT claim-text (#PCDATA)>
<!ELEMENT p (#PCDATA)>


In [6]:
# Emit an ST.36-style instance from a real corpus patent, then validate it against the DTD.
docs = bs.ensure("docs_canonical")
d = next(x for x in docs if x.doc_id == "US9081550B2")
xml_bytes = P.patentdoc_to_st36_xml(d)
print(xml_bytes.decode("utf-8")[:520], "...\n")
ok, errors = P.validate_dtd(xml_bytes, P.ST36_DTD)
print("DTD validation:", "VALID" if ok else "INVALID", "| errors:", errors)

# Failure mode: violate the content model (claim without required claim-text) -> invalid.
broken = xml_bytes.replace(b"<claim-text>", b"<oops>").replace(b"</claim-text>", b"</oops>")
ok_b, err_b = P.validate_dtd(broken, P.ST36_DTD)
print("broken instance:", "VALID" if ok_b else "INVALID", "|", err_b[0][:80] if err_b else "")

<patent-document lang="en" country="US" doc-number="US9081550B2" kind="B2">
  <bibliographic-data>
    <invention-title>Adding speech capabilities to existing computer applications with complex graphical user interfaces</invention-title>
    <publication-date>2015-07-14</publication-date>
  </bibliographic-data>
  <abstract>
    <p>At design time of a graphical user interface (GUI), a software component (VUIcontroller) is added to the GUI. At run time of the GUI, the VUIcontroller analyzes the GUI from within a pro ...

DTD validation: VALID | errors: []
broken instance: INVALID | <string>:10:0:ERROR:VALID:DTD_CONTENT_MODEL: Element claim content does not foll


## 5–6. WIPO ST.36 vs ST.96

**ST.36** (legacy) organizes patent XML with **DTDs**; **ST.96** (current) uses **XML Schema
(XSD)** with real namespaces, strong typing, and componentized schemas (Common + Patent + …).
The current ST.96 version is **v9.0** (approved by the WIPO CWS XML4IP Task Force on
2025-04-01). Legacy corpora still ship ST.36, so an ingestion pipeline must handle both.

### XSD validation against the genuine WIPO schema — and a version-drift lesson

Our instance is WIPO's official example, labelled `st96Version="V8_0"`. Validating it against
the genuine **v9.0** XSD surfaces real, explainable errors (the version is a *fixed* attribute;
MathML's namespace changed). This is exactly what a pipeline hits when a standard increments.

In [7]:
from lxml import etree
sch = P.extract_st96_schema(bs.ARTIFACTS / "st96_v9", bs.DATA / "schema" / "ST96XMLSchema_V9_0_Flattened.zip")
xsd_path = sch / "PatentPublication_V9_0.xsd"

ok, errors = P.validate_xsd(tree, xsd_path)
print("genuine V8 instance vs V9.0 schema ->", "VALID" if ok else "INVALID")
for e in errors[:3]:
    print("   •", e[:110])

genuine V8 instance vs V9.0 schema -> INVALID
   • 2: Element '{http://www.wipo.int/standards/XMLSchema/ST96/Patent}PatentPublication', attribute '{http://www.wi
   • 4: Element '{http://www.wipo.int/standards/XMLSchema/ST96/Patent}BibliographicData', attribute '{http://www.wi
   • 1138: Element '{http://www.w3.org/1998/Math/MathML3}math': This element is not expected. Expected is one of ( 


In [8]:
# Migrate V8 -> V9 (bump the fixed version attributes + upgrade the MathML namespace), re-validate.
raw = (bs.DATA / "xml" / "ST96_PatentPublication_Example.xml").read_bytes()
migrated = P.migrate_st96_version(raw, "V9_0")
tree_v9 = etree.ElementTree(etree.fromstring(migrated))
ok2, errors2 = P.validate_xsd(tree_v9, xsd_path)
print("after migration vs V9.0 schema ->", "VALID" if ok2 else "INVALID", "| remaining errors:", len(errors2))

after migration vs V9.0 schema -> VALID | remaining errors: 0


### ST.36 vs ST.96 comparison

| Dimension | **ST.36** (legacy) | **ST.96** (current, v9.0) |
|---|---|---|
| Schema technology | DTD | XML Schema (XSD) |
| Namespaces | None | Yes (`pat:`, `com:`, …) |
| Typing | Weak (mostly `#PCDATA`) | Strong (dates, codes, enums) |
| Validation | Structural only | Structural **+** datatype + fixed-value |
| Extensibility | Limited | Componentized, versioned |
| Role today | Legacy corpora, some office pipelines | Current exchange standard |
| Ingestion implication | Tolerant parsing, few guarantees | Validate + normalize + handle version drift |

Newer WIPO work (e.g. richer data models) exists, but **ST.36/ST.96 are what patent XML
corpora actually contain** — the practical ingestion targets.

## Chapter invariants

In [9]:
assert bib["publication_number"] == "08936998" and bib["kind_code"] == "B2"
assert bib["st96_version"] == "V8_0"
assert len(claims) >= 1
assert P.validate_dtd(xml_bytes, P.ST36_DTD)[0]                 # real content validates against DTD
assert not P.validate_xsd(tree, xsd_path)[0]                    # V8 fails against V9 (drift)
assert P.validate_xsd(tree_v9, xsd_path)[0]                     # migration fixes it
print("All Chapter 02 invariants hold.")

All Chapter 02 invariants hold.


In [10]:
# === Chapter 02 validation footer ===
import time, platform, sys, importlib.metadata as _md
_pkgs = ['lxml']
print("Chapter 02 — environment")
print("  Python :", sys.version.split()[0], "on", platform.system(), platform.release())
for _p in _pkgs:
    try: print(f"  {_p:24}: {_md.version(_p)}")
    except Exception: print(f"  {_p:24}: (not installed)")
print()
print("CHAPTER 02 VALIDATION: PASS")

Chapter 02 — environment
  Python : 3.12.10 on Windows 11
  lxml                    : 6.1.1

CHAPTER 02 VALIDATION: PASS
